Install request

In [23]:
#%pip install requests

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 30, Finished, Available, Finished, False)

Imports libraries

In [2]:
import requests, json
from datetime import datetime
from pyspark.sql.functions import *

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 9, Finished, Available, Finished, False)

Configuration for FHIR API

In [3]:
BASE_URL = "https://hapi.fhir.org/baseR4"
RESOURCES = ["Patient", "Encounter", "Observation", "Condition"]

PAGE_SIZE = 50
RAW_PATH = "Files/raw"

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 10, Finished, Available, Finished, False)

API Fetch with Pagination

In [4]:
def fetch_fhir(resource):
    url = f"{BASE_URL}/{resource}?_count={PAGE_SIZE}"
    all_data = []

    while url:
        response = requests.get(url)
        data = response.json()

        all_data.append(data)

        # Pagination handling
        next_url = None
        if "link" in data:
            for l in data["link"]:
                if l["relation"] == "next":
                    next_url = l["url"]

        url = next_url

    return all_data

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 11, Finished, Available, Finished, False)

RAW INGESTION

In [5]:
def load_raw(resource):
    data = fetch_fhir(resource)

    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    date_part = datetime.now().strftime("%Y-%m-%d")

    df = spark.createDataFrame([(json.dumps(d),) for d in data], ["raw_json"]) \
        .withColumn("extraction_timestamp", lit(ts)) \
        .withColumn("load_date", current_date()) \
        .withColumn("resource", lit(resource)) \
        .withColumn("api_url", lit(f"{BASE_URL}/{resource}"))

    path = f"{RAW_PATH}/{resource.lower()}/date={date_part}"

    df.write.mode("append").json(path)

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 12, Finished, Available, Finished, False)

DATA INGESTION

In [6]:
for r in RESOURCES:
    load_raw(r)

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 13, Finished, Available, Finished, False)

Create Bronze DB

In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_db")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 14, Finished, Available, Finished, False)

DataFrame[]

RAW TO BRONZE

In [8]:
def load_bronze(resource):
    df = spark.read.json(f"{RAW_PATH}/{resource.lower()}/")

    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze_db.{resource.lower()}")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 15, Finished, Available, Finished, False)

Execute Bronze Load

In [9]:
for r in RESOURCES:
    load_bronze(r)

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 16, Finished, Available, Finished, False)

SILVER LAYER (Flattening)

In [12]:
df = spark.read.table("bronze_db.patient")

schema = spark.read.json(df.select("raw_json").rdd.map(lambda r: r[0])).schema

parsed = df.withColumn("j", from_json(col("raw_json"), schema))

# explode entry
exploded = parsed.withColumn("entry_exp", explode_outer(col("j.entry")))

# extract resource
res = exploded.withColumn("res", col("entry_exp.resource"))

final = res.select(
    col("res.id").alias("patient_id"),
    col("res.gender"),
    col("res.birthDate"),
    col("res.name"),
    col("extraction_timestamp")
)

final = final.withColumn("name_exp", explode_outer("name")) \
    .select(
        "patient_id",
        "gender",
        "birthDate",
        col("name_exp.family").alias("last_name"),
        col("name_exp.given")[0].alias("first_name"),
        "extraction_timestamp"
    ).dropDuplicates(["patient_id"])

final.write.format("delta").mode("overwrite").saveAsTable("silver_patient")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 19, Finished, Available, Finished, False)

In [13]:
df = spark.read.table("bronze_db.encounter")

schema = spark.read.json(df.select("raw_json").rdd.map(lambda r: r[0])).schema

parsed = df.withColumn("j", from_json(col("raw_json"), schema))

exploded = parsed.withColumn("entry_exp", explode_outer(col("j.entry")))

res = exploded.withColumn("res", col("entry_exp.resource"))

enc = res.select(
    col("res.id").alias("encounter_id"),
    col("res.status"),
    col("res.subject.reference").alias("patient_reference"),
    col("res.period.start").alias("start_time"),
    col("res.period.end").alias("end_time"),
    col("extraction_timestamp")
).dropDuplicates(["encounter_id"])

enc.write.format("delta").mode("overwrite").saveAsTable("silver_encounter")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 20, Finished, Available, Finished, False)

In [14]:
df = spark.read.table("bronze_db.observation")

schema = spark.read.json(df.select("raw_json").rdd.map(lambda r: r[0])).schema

parsed = df.withColumn("j", from_json(col("raw_json"), schema))

exploded = parsed.withColumn("entry_exp", explode_outer(col("j.entry")))

res = exploded.withColumn("res", col("entry_exp.resource"))

obs = res.select(
    col("res.id").alias("observation_id"),
    col("res.code.text").alias("type"),
    col("res.valueQuantity.value").alias("value"),
    col("res.valueQuantity.unit").alias("unit"),
    col("res.subject.reference").alias("patient_reference"),
    col("extraction_timestamp")
).dropDuplicates(["observation_id"])

obs.write.format("delta").mode("overwrite").saveAsTable("silver_observation")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 21, Finished, Available, Finished, False)

In [15]:
df = spark.read.table("bronze_db.condition")

schema = spark.read.json(df.select("raw_json").rdd.map(lambda r: r[0])).schema

parsed = df.withColumn("j", from_json(col("raw_json"), schema))

exploded = parsed.withColumn("entry_exp", explode_outer(col("j.entry")))

res = exploded.withColumn("res", col("entry_exp.resource"))

cond = res.select(
    col("res.id").alias("condition_id"),
    col("res.code.text").alias("condition"),
    col("res.subject.reference").alias("patient_reference"),
    col("extraction_timestamp")
).dropDuplicates(["condition_id"])

cond.write.format("delta").mode("overwrite").saveAsTable("silver_condition")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 22, Finished, Available, Finished, False)

GOLD layer

In [16]:
spark.sql("""
CREATE OR REPLACE TABLE gold_patient AS
SELECT patient_id, first_name, last_name, gender, birthDate
FROM silver_patient
""")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 23, Finished, Available, Finished, False)

DataFrame[]

In [17]:
spark.sql("""
CREATE OR REPLACE TABLE gold_encounter AS
SELECT encounter_id, patient_reference, start_time, end_time, status
FROM silver_encounter
""")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 24, Finished, Available, Finished, False)

DataFrame[]

SCD TYPE 2

In [18]:
spark.sql("""
CREATE TABLE IF NOT EXISTS gold_patient_scd2 (
patient_id STRING,
first_name STRING,
last_name STRING,
gender STRING,
birthDate STRING,
effective_from TIMESTAMP,
effective_to TIMESTAMP,
is_current BOOLEAN
)
USING DELTA
""")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 25, Finished, Available, Finished, False)

DataFrame[]

In [19]:
spark.sql("""
MERGE INTO gold_patient_scd2 t
USING (
  SELECT *,
  current_timestamp() as effective_from,
  null as effective_to,
  true as is_current
  FROM gold_patient
) s
ON t.patient_id = s.patient_id AND t.is_current = true

WHEN MATCHED AND (
  t.first_name <> s.first_name OR
  t.last_name <> s.last_name OR
  t.gender <> s.gender OR
  t.birthDate <> s.birthDate
)
THEN UPDATE SET
  t.effective_to = current_timestamp(),
  t.is_current = false

WHEN NOT MATCHED THEN
INSERT *
""")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 26, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Monitor Logging

In [22]:
from datetime import datetime

log_df = spark.createDataFrame([
    ("FHIR_PIPELINE", datetime.now(), "SUCCESS")
], ["process", "timestamp", "status"])

log_df.write.mode("append").saveAsTable("pipeline_logs")

StatementMeta(, df085a2f-183f-44fb-a863-10267afd47f3, 29, Finished, Available, Finished, False)